# AlphaForge — EDA
Exploratory Data Analysis of raw OHLCV data across all assets.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from src.data.storage import load_ohlcv
from src.config import get_settings

settings = get_settings()
print('Assets:', settings.all_assets)

In [ ]:
# Load BTC data
btc = load_ohlcv('BTC/USDT', '1h')
print(f'BTC rows: {len(btc):,}')
print(f'Date range: {btc.index.min()} → {btc.index.max()}')
btc.describe()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10))
fig.suptitle('BTC/USDT — Price, Volume & Returns', fontsize=14)

# Price
axes[0].plot(btc.index, btc['close'], color='#00d4ff', linewidth=0.8)
axes[0].set_title('Close Price')
axes[0].set_facecolor('#0b1020')

# Volume
axes[1].bar(btc.index, btc['volume'], color='#10b981', alpha=0.7, width=0.04)
axes[1].set_title('Volume')
axes[1].set_facecolor('#0b1020')

# Returns distribution
returns = btc['close'].pct_change().dropna()
axes[2].hist(returns, bins=100, color='#7c3aed', alpha=0.8, edgecolor='none')
axes[2].axvline(returns.mean(), color='#f59e0b', linestyle='--', label=f'Mean: {returns.mean():.4f}')
axes[2].set_title('1h Return Distribution')
axes[2].legend()
axes[2].set_facecolor('#0b1020')

for ax in axes:
    ax.tick_params(colors='#94a3b8')

plt.tight_layout()
plt.savefig('../notebooks/btc_eda.png', dpi=150, bbox_inches='tight', facecolor='#050810')
plt.show()

In [ ]:
# Cross-asset correlation
all_returns = {}
for asset in settings.all_assets:
    try:
        tf = '1h' if '/' in asset else '1d'
        df = load_ohlcv(asset, tf)
        all_returns[asset] = df['close'].pct_change()
    except Exception as e:
        print(f'Could not load {asset}: {e}')

returns_df = pd.DataFrame(all_returns).dropna()
corr = returns_df.corr()

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr.values, cmap='RdYlGn', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45)
ax.set_yticklabels(corr.columns)
plt.colorbar(im, ax=ax)
ax.set_title('Cross-Asset Return Correlations')
plt.tight_layout()
plt.show()
print(corr.round(3))

In [ ]:
# Stationarity tests (ADF) — important for time-series ML
from statsmodels.tsa.stattools import adfuller

print('ADF Test Results (H0: series has unit root / is non-stationary)')
print('-' * 60)
for asset, ret in all_returns.items():
    ret_clean = ret.dropna()
    result = adfuller(ret_clean, autolag='AIC')
    p = result[1]
    status = '✅ Stationary' if p < 0.05 else '❌ Non-stationary'
    print(f'{asset:12s} ADF stat={result[0]:.3f}  p={p:.4f}  {status}')